# Parent RAG - "Os Sertões" de Euclides da Cunha

Implementação de **Parent Document Retriever**, que melhora a qualidade do retrieval através de uma estratégia de dois níveis:

- **Child chunks** (pequenos): usados para busca por similaridade — chunks menores capturam melhor a semântica específica.
- **Parent chunks** (grandes): retornados como contexto para a LLM — preservam mais informação e coerência textual.

Isso resolve o dilema do Naive RAG: chunks pequenos encontram melhor, mas chunks grandes geram melhor.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no .env ou como variável de ambiente"

## 1. Carregamento do PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "os-sertoes.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Total de páginas carregadas: {len(pages)}")

## 2. Configuração dos Splitters (Parent e Child)

- **Parent splitter**: chunks grandes (~2000 chars) que mantêm contexto amplo
- **Child splitter**: chunks pequenos (~400 chars) que permitem busca mais precisa

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

parent_chunks = parent_splitter.split_documents(pages)
print(f"Parent chunks: {len(parent_chunks)}")
print(f"Tamanho médio parent: {sum(len(c.page_content) for c in parent_chunks) / len(parent_chunks):.0f} chars")

## 3. Embeddings e Parent Document Retriever

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

CHROMA_DIR = "./chroma_parent"

child_vectorstore = Chroma(
    collection_name="os_sertoes_child",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

docstore = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=child_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 10}
)

print("Indexando documentos (child + parent)... Isso pode levar alguns minutos.")
parent_retriever.add_documents(pages)

print(f"\nChild chunks no vectorstore: {child_vectorstore._collection.count()}")
print(f"Parent chunks no docstore: {len(list(docstore.yield_keys()))}")

## 4. Verificação do Mecanismo Parent

Vamos verificar que a busca por child chunks retorna parent chunks maiores.

In [ ]:
query_teste = "descrição do sertão nordestino"

child_results = child_vectorstore.similarity_search(query_teste, k=3)
print("=== CHILD CHUNKS (busca direta no vectorstore) ===")
for i, doc in enumerate(child_results, 1):
    print(f"\nChild {i} ({len(doc.page_content)} chars): {doc.page_content[:150]}...")

print("\n" + "="*80)

parent_results = parent_retriever.invoke(query_teste)
print("\n=== PARENT CHUNKS (retornados pelo retriever) ===")
for i, doc in enumerate(parent_results, 1):
    print(f"\nParent {i} ({len(doc.page_content)} chars): {doc.page_content[:150]}...")

print(f"\nObs: Child chunks têm ~400 chars, Parent chunks têm ~2000 chars")

## 5. Configuração da Chain RAG

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=2000
)

PROMPT_TEMPLATE = """
Você é um especialista em literatura brasileira, especificamente na obra "Os Sertões" de Euclides da Cunha.
Responda a pergunta com base EXCLUSIVAMENTE nos trechos do livro fornecidos abaixo.
Os trechos abaixo são seções amplas do livro que fornecem contexto completo.
Se a informação não estiver nos trechos, diga que não encontrou a informação no contexto fornecido.
Cite trechos relevantes quando possível.

Contexto:
{context}

Pergunta: {question}

Resposta detalhada:
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Trecho {i+1} — Pág. {d.metadata.get('page', '?')}]\n{d.page_content}"
        for i, d in enumerate(docs)
    )

parent_rag_chain = (
    {"context": parent_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Pipeline Parent RAG configurado com sucesso!")

## 6. Execução das Perguntas

In [ ]:
PERGUNTAS = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em 'Os Sertões'? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

In [ ]:
respostas_parent = {}

for i, pergunta in enumerate(PERGUNTAS, 1):
    print(f"{'='*80}")
    print(f"PERGUNTA {i}: {pergunta}")
    print(f"{'='*80}")
    
    resposta = parent_rag_chain.invoke(pergunta)
    respostas_parent[i] = resposta
    
    print(f"\nRESPOSTA:\n{resposta}")
    print()

## 7. Análise dos Chunks Recuperados

Verificando a diferença entre child e parent retrieval.

In [ ]:
for i, pergunta in enumerate(PERGUNTAS, 1):
    print(f"\n{'='*80}")
    print(f"PERGUNTA {i}: {pergunta[:80]}...")
    print(f"{'='*80}")
    
    parent_docs = parent_retriever.invoke(pergunta)
    
    print(f"\nRetornados {len(parent_docs)} parent chunks")
    for j, doc in enumerate(parent_docs, 1):
        print(f"  Parent {j}: {len(doc.page_content)} chars | Pág. {doc.metadata.get('page', '?')} | {doc.page_content[:100]}...")